# 03 · Two variables, two questions

## Context

PRISM minimum and maximum temperature share one coordinate system, so we can
ask related questions without juggling separate grids.

## Question

Where was maximum temperature unusual on the coldest regional day, and how did
the day-to-night temperature range change through January?

## Analysis story

We validate the physical relationship between variables, then answer a spatial
and a temporal question with two readable pipes.


### Data used in this lesson

Every value comes from the PRISM Group at Oregon State University's AN91d
daily 4 km climate product. This repository carries a small Boulder-region
extract for 1–30 January 2024 so the lesson runs offline without replacing
observations with generated values. The [data validation page](../validation/data.md)
records source URLs, terms, checksums, bounds, units, and acceptance tests.

## Prepare · Validate the shared dataset

In [ ]:
from pathlib import Path

import xarray as xr

# Find the repository from either a root-level documentation build or a kernel
# started beside this notebook, then open the checksum-controlled PRISM extract.
data_path = next(
    candidate / "data" / "vignettes" / "prism_boulder_january_2024.nc"
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "data" / "vignettes" / "prism_boulder_january_2024.nc").exists()
)
prism = xr.open_dataset(data_path, engine="scipy").load()

# These assertions are part of the teaching contract: official source,
# canonical cube dimensions, complete daily time, and declared Celsius units.
assert prism.attrs["source"] == "PRISM Group, Oregon State University"
assert prism.attrs["is_synthetic"] == 0
assert prism.sizes == {"time": 30, "y": 24, "x": 24}
assert prism["tmax"].attrs["units"] == "degC"

import numpy as np

# Minimum temperature cannot exceed maximum temperature, and the derived range
# must be exactly traceable to those two observed PRISM variables.
assert bool((prism["tmin"] <= prism["tmax"]).all())
np.testing.assert_allclose(
    prism["diurnal_range"], prism["tmax"] - prism["tmin"], rtol=0, atol=1e-5
)
coldest_day = prism["tmax"].mean(("y", "x")).argmin("time")

## Pipes · Express each analysis as one sentence

In [ ]:
from cubedynamics import pipe, verbs as v

maximum_temperature_anomaly = (
    pipe(prism["tmax"])
    | v.anomaly(dim="time")
).unwrap()

regional_diurnal_range = (
    pipe(prism["diurnal_range"])
    | v.mean(dim=("y", "x"))
).unwrap()

## Figure · Put the spatial and temporal answers together

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), constrained_layout=True)
maximum_temperature_anomaly.isel(time=coldest_day).plot(
    ax=axes[0], cmap="RdBu_r", center=0, cbar_kwargs={"label": "Anomaly (°C)"}
)
date = str(prism.time.isel(time=coldest_day).values)[:10]
axes[0].set_title(f"Maximum-temperature anomaly · {date}")
regional_diurnal_range.plot(ax=axes[1], marker="o", color="#775a3a")
axes[1].set_title("Boulder-region diurnal temperature range")
axes[1].set_ylabel("Daily range (°C)")
plt.show()

## What the figure tells us

The coldest regional day was not equally unusual everywhere, while the
day-to-night range followed a separate trajectory.

## Try the next variation

Pipe `tmin` through the same anomaly verb and compare its spatial pattern.